# NIFTY Morning Signal

**Two-phase workflow:**

| Phase | When | Cells to run | What you get |
|---|---|---|---|
| Pre-market | Before 9:15 AM | 1 → 2 → 3 → 4 | Signal from overnight data (no gap yet) |
| Post-open  | At/after 9:15 AM | 5 → 6 | Final signal including actual gap |

No manual input needed for Phase 1. Only Cell 5 requires one number.

In [ ]:
# ── Cell 1: Imports & Config (run once) ───────────────────────────────────────
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import date, timedelta
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE        = Path.cwd()
SIGNALS_CSV = BASE / 'v2' / 'v2_reliable_signals.csv'

# ── Thresholds — must match py_backtest.py exactly ────────────────────────────
GAP_THRESHOLD       = 0.0015   # 0.15%  gap up/down
GAP_LARGE_THRESHOLD = 0.0050   # 0.50%  strong gap  ← matches training threshold
VIX_RISING_THRESHOLD = 0.03    # 3%     VIX rising  ← matches training threshold
VIX_SPIKE_THRESHOLD = 0.05     # 5%     VIX spike
BASE_RATE           = 54.5     # historical base rate: 54.5% of sessions are DOWN

# ── Trade parameters — matches best backtest result ───────────────────────────
STOP_LOSS_PCT      = 0.40      # exit if option premium drops 40%
PROFIT_TARGET_PCT  = 0.40      # exit if option premium rises 40%

# ── Signal mode & lot caps — must match py_backtest.py ───────────────────────
SIGNAL_MODE   = "BEARISH_ONLY"   # "ALL" | "BULLISH_ONLY" | "BULLISH_ONLY"
DTE0_MAX_LOTS = 10               # max lots on expiry day (DTE=0)
MAX_LOTS      = 20               # hard global cap

W = 70   # output width

print('Config loaded.')
print(f'  SL={STOP_LOSS_PCT:.0%} of premium  |  TP={PROFIT_TARGET_PCT:.0%} of premium')
print(f'  Gap Large = {GAP_LARGE_THRESHOLD:.2%}  |  VIX Rising = >{VIX_RISING_THRESHOLD:.0%}')
print(f'  Signal mode : {SIGNAL_MODE}  |  DTE0 cap: {DTE0_MAX_LOTS}  |  Global cap: {MAX_LOTS}')

Config loaded.
  SL=40% of premium  |  TP=40% of premium
  Gap Large = 0.50%  |  VIX Rising = >3%
  Signal mode : BEARISH_ONLY  |  DTE0 cap: 10  |  Global cap: 20


In [ ]:
# ── Cell 2: Fetch global market data — FULLY AUTOMATIC ────────────────────────
today = date.today()

# ── Weekend / holiday guard ────────────────────────────────────────────────────
if today.weekday() >= 5:
    print(f'WARNING: Today is {today.strftime("%A")} — market is closed. Nothing to do.')
    raise SystemExit('Weekend — stopping execution.')

start = str(today - timedelta(days=12))
end   = str(today + timedelta(days=1))

print(f'Fetching market data for {today} ...', end=' ', flush=True)

TICKERS = {
    'SP500'  : '^GSPC',
    'SGX'    : 'NKD=F',
    'DAX'    : '^GDAXI',
    'VIX_US' : '^VIX',
    'NIFTY'  : '^NSEI',
}

raw = {}
fetch_errors = []
for name, ticker in TICKERS.items():
    try:
        df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
        df = df[df.index.date < today].copy()
        raw[name] = df
    except Exception as e:
        raw[name] = pd.DataFrame()
        fetch_errors.append(f'{name}: {e}')

print('done.')
if fetch_errors:
    print(f'  WARNING: fetch errors for: {", ".join(fetch_errors)}')

# ── Helper functions ───────────────────────────────────────────────────────────
MAX_STALE_DAYS = 5   # if most recent data is older than this, treat as missing

def _ret(name):
    df = raw[name]
    if len(df) < 2:
        return None
    # Staleness check
    last_date = df.index[-1].date()
    if (today - last_date).days > MAX_STALE_DAYS:
        return None
    return float((df['Close'].iloc[-1] - df['Close'].iloc[-2]) / df['Close'].iloc[-2])

def _close(name):
    df = raw[name]
    if len(df) == 0:
        return None
    last_date = df.index[-1].date()
    if (today - last_date).days > MAX_STALE_DAYS:
        return None
    return float(df['Close'].iloc[-1])

sp500_ret   = _ret('SP500')
sgx_ret     = _ret('SGX')
dax_ret     = _ret('DAX')
vix_ret     = _ret('VIX_US')
nifty_ret   = _ret('NIFTY')
nifty_close = _close('NIFTY')
vix_level   = _close('VIX_US')

# ── Indian market holiday check ────────────────────────────────────────────────
if nifty_close is None:
    print()
    print('WARNING: NIFTY data unavailable.')
    print('  Possible reasons: Indian market holiday, yfinance outage, or data delay.')
    print('  No trade today.')
    raise SystemExit('NIFTY data unavailable — stopping execution.')

# ── Data quality warnings ──────────────────────────────────────────────────────
missing = [k for k, v in {'SP500': sp500_ret, 'SGX': sgx_ret, 'DAX': dax_ret, 'VIX': vix_ret}.items() if v is None]
if missing:
    print(f'  WARNING: Missing/stale data for: {", ".join(missing)}')
    print(f'  These signals will default to False. Signal accuracy may be reduced.')

print()
print(f'  NIFTY prev close   : {nifty_close:>10,.1f}')
print(f'  S&P 500 yesterday  : {sp500_ret:>+10.2%}' if sp500_ret is not None else '  S&P 500 yesterday  :    MISSING')
print(f'  SGX/Nikkei futures : {sgx_ret:>+10.2%}'   if sgx_ret   is not None else '  SGX/Nikkei futures :    MISSING')
print(f'  DAX yesterday      : {dax_ret:>+10.2%}'   if dax_ret   is not None else '  DAX yesterday      :    MISSING')
print(f'  VIX yesterday      : {vix_ret:>+10.2%}  (level: {vix_level:.1f})' if vix_ret is not None and vix_level is not None else '  VIX yesterday      :    MISSING')
print(f'  Prev India return  : {nifty_ret:>+10.2%}' if nifty_ret is not None else '  Prev India return  :    MISSING')

In [ ]:
# ── Cell 3: Compute overnight signals (no gap yet) ────────────────────────────
# nifty_close is guaranteed non-None here (Cell 2 halts otherwise)

signals = {
    'Gap Up'          : False,   # unknown pre-market
    'Gap Up Strong'   : False,
    'Gap Down'        : False,
    'Prev India UP'   : nifty_ret is not None and nifty_ret  > 0,
    'Prev India DOWN' : nifty_ret is not None and nifty_ret  < 0,
    'US UP'           : sp500_ret is not None and sp500_ret  > 0,
    'US DOWN'         : sp500_ret is not None and sp500_ret  < 0,
    'SGX UP'          : sgx_ret   is not None and sgx_ret    > 0,
    'SGX DOWN'        : sgx_ret   is not None and sgx_ret    < 0,
    'DAX UP'          : dax_ret   is not None and dax_ret    > 0,
    'VIX Rising'      : vix_ret   is not None and vix_ret    > VIX_RISING_THRESHOLD,
    'VIX Falling'     : vix_ret   is not None and vix_ret    < 0,
    'VIX Spike'       : vix_ret   is not None and vix_ret    > VIX_SPIKE_THRESHOLD,
}

active_overnight = [k for k, v in signals.items() if v]
missing_signals  = []
if nifty_ret  is None: missing_signals.append('Prev India ret')
if sp500_ret  is None: missing_signals.append('US')
if sgx_ret    is None: missing_signals.append('SGX')
if dax_ret    is None: missing_signals.append('DAX')
if vix_ret    is None: missing_signals.append('VIX')

print('Overnight signals active:')
for s in active_overnight:
    print(f'  + {s}')
if not active_overnight:
    print('  (none active)')
if missing_signals:
    print(f'\n  NOTE: {", ".join(missing_signals)} data missing — those signals default to False.')

In [ ]:
# ── Cell 4: PRE-MARKET SIGNAL (run before 9:15 AM) ────────────────────────────
# Uses the same top-3 bearish + top-3 bullish as the backtest.
# Gap-based combos are excluded here (gap is unknown pre-market).

if not SIGNALS_CSV.exists():
    raise FileNotFoundError(f'{SIGNALS_CSV} not found. Run v2/v2_india_global.ipynb first.')

reliable = pd.read_csv(SIGNALS_CSV)

# ── Replicate py_backtest.py signal selection exactly ─────────────────────────
top_bearish = (reliable[reliable['P_Down'] > BASE_RATE]
               .sort_values('Edge', ascending=False)
               .head(3)
               .reset_index(drop=True))

top_bullish = (reliable[reliable['P_Down'] < (100 - BASE_RATE)]
               .sort_values('P_Down', ascending=True)
               .head(3)
               .reset_index(drop=True))

bear_combos = list(top_bearish['Signal'])
bull_combos = list(top_bullish['Signal'])

GAP_SIGNALS = {'Gap Up', 'Gap Up Strong', 'Gap Down'}

def _has_gap(combo_str):
    return any(g in [s.strip() for s in combo_str.split('+')] for g in GAP_SIGNALS)

def _combo_fires(combo_str):
    return all(signals.get(s.strip(), False) for s in combo_str.split('+'))

# Pre-market: only non-gap combos can fire; SIGNAL_MODE suppresses a direction
bear_fired = ([c for c in bear_combos if not _has_gap(c) and _combo_fires(c)]
              if SIGNAL_MODE != "BULLISH_ONLY" else [])
bull_fired  = ([c for c in bull_combos if not _has_gap(c) and _combo_fires(c)]
               if SIGNAL_MODE != "BEARISH_ONLY" else [])

# ── Print helper ───────────────────────────────────────────────────────────────
def _print_signal(bear_fired, bull_fired, label, show_top3=True):
    if show_top3:
        if SIGNAL_MODE != "BULLISH_ONLY":
            print('  Top-3 bearish combos being monitored:')
            for c in bear_combos:
                tag = '✓ ACTIVE' if c in bear_fired else '✗'
                print(f'    {tag}  {c}')
        else:
            print('  Bearish combos: [suppressed — BULLISH_ONLY mode]')
        if SIGNAL_MODE != "BEARISH_ONLY":
            print('  Top-3 bullish combos being monitored:')
            for c in bull_combos:
                tag = '✓ ACTIVE' if c in bull_fired else '✗'
                print(f'    {tag}  {c}')
        else:
            print('  Bullish combos: [suppressed — BEARISH_ONLY mode]')
        print()

    print('=' * W)
    if not bear_fired and not bull_fired:
        print(f'  {label}:  NEUTRAL  —  SIT OUT')
        print(f'  None of the top-3 bear/bull combos are active.')
    elif bear_fired and not bull_fired:
        row = top_bearish[top_bearish['Signal'] == bear_fired[0]].iloc[0]
        print(f'  {label}:  ▼  BEARISH  (Buy 1-OTM PUT)')
        print(f'  Combo  : {bear_fired[0]}')
        print(f'  P(DOWN): {row["P_Down"]:.1f}%  |  Edge: +{row["Edge"]:.1f}%  |  {row["Sig"]}  N={int(row["N"])}')
    elif bull_fired and not bear_fired:
        row = top_bullish[top_bullish['Signal'] == bull_fired[0]].iloc[0]
        print(f'  {label}:  ▲  BULLISH  (Buy 1-OTM CALL)')
        print(f'  Combo  : {bull_fired[0]}')
        print(f'  P(UP)  : {100-row["P_Down"]:.1f}%  |  Edge: +{abs(row["Edge"]):.1f}%  |  {row["Sig"]}  N={int(row["N"])}')
    else:
        print(f'  {label}:  ⚡ CONFLICT  —  SIT OUT  (both bear & bull fire)')
    print('=' * W)

print('=' * W)
print(f"  PRE-MARKET  ·  {today.strftime('%d %b %Y, %A')}  ·  (gap unknown)")
if SIGNAL_MODE != 'ALL':
    print(f"  [Mode: {SIGNAL_MODE} — {'bullish' if SIGNAL_MODE == 'BEARISH_ONLY' else 'bearish'} signals suppressed]")
print('=' * W)
print()
_print_signal(bear_fired, bull_fired, label='PRE-MARKET SIGNAL')
print()
print('  → Wait for 9:15 AM, enter actual NIFTY open in Cell 5, then run Cell 6.')

In [ ]:
# ── Cell 5: MANUAL INPUT — enter actual NIFTY open at/after 9:15 AM ──────────
#
#  Look at the NIFTY 50 index on your broker screen the moment market opens.
#  Use the price at 9:15 AM (first candle open).
#
NIFTY_ACTUAL_OPEN = 22390    # ← ENTER ACTUAL NIFTY OPEN PRICE HERE
#
# ─────────────────────────────────────────────────────────────────────────────
print(f'Actual NIFTY open set to: {NIFTY_ACTUAL_OPEN:,.0f}')
print('Run Cell 6 for the final signal.')

In [ ]:
# ── Cell 6: FINAL SIGNAL — includes actual gap (run at/after 9:15 AM) ─────────
if not NIFTY_ACTUAL_OPEN or NIFTY_ACTUAL_OPEN <= 0:
    raise ValueError('NIFTY_ACTUAL_OPEN is not set. Enter the actual opening price in Cell 5.')

if nifty_close is None or nifty_close == 0:
    raise ValueError('NIFTY previous close unavailable — cannot compute gap.')

gap = (NIFTY_ACTUAL_OPEN - nifty_close) / nifty_close

# Sanity check: gap > 5% is almost certainly a data entry error
if abs(gap) > 0.05:
    print(f'WARNING: Gap of {gap:+.2%} is unusually large.')
    print(f'  Please verify NIFTY_ACTUAL_OPEN={NIFTY_ACTUAL_OPEN:,.0f} and prev close={nifty_close:,.0f}')
    print(f'  Proceeding anyway — double-check before trading.')

if gap > GAP_LARGE_THRESHOLD:
    gap_label = f'GAP UP STRONG ({gap:+.2%})'
elif gap > GAP_THRESHOLD:
    gap_label = f'GAP UP ({gap:+.2%})'
elif gap < -GAP_THRESHOLD:
    gap_label = f'GAP DOWN ({gap:+.2%})'
else:
    gap_label = f'FLAT ({gap:+.2%}) — no gap signal'

signals['Gap Up']        = gap >  GAP_THRESHOLD
signals['Gap Up Strong'] = gap >  GAP_LARGE_THRESHOLD
signals['Gap Down']      = gap < -GAP_THRESHOLD

all_active = [k for k, v in signals.items() if v]
print(f'Gap : {NIFTY_ACTUAL_OPEN:,.0f} vs prev close {nifty_close:,.0f}  ->  {gap_label}')
print(f'Active signals: {", ".join(all_active) if all_active else "none"}')
print()

bear_fired_final = ([c for c in bear_combos if _combo_fires(c)]
                    if SIGNAL_MODE != "BULLISH_ONLY" else [])
bull_fired_final = ([c for c in bull_combos  if _combo_fires(c)]
                    if SIGNAL_MODE != "BEARISH_ONLY" else [])

# ── DTE & lot cap calculation ─────────────────────────────────────────────────
def _nearest_thursday(d):
    days_ahead = (3 - d.weekday()) % 7
    return d + timedelta(days=days_ahead)

expiry         = _nearest_thursday(today)
dte            = (expiry - today).days
applicable_cap = DTE0_MAX_LOTS if dte == 0 else MAX_LOTS

print('=' * W)
print(f"  FINAL SIGNAL  .  {today.strftime('%d %b %Y, %A')}  .  Open: {NIFTY_ACTUAL_OPEN:,.0f}  ({gap:+.2%})")
if SIGNAL_MODE != 'ALL':
    print(f"  [Mode: {SIGNAL_MODE}]")
print('=' * W)
print()
_print_signal(bear_fired_final, bull_fired_final, label='FINAL SIGNAL')

print()
print('  EXECUTION GUIDE:')
print(f'    Expiry     : {expiry.strftime("%d %b %Y")} (DTE={dte})')
print(f'    Lot cap    : max {applicable_cap} lots{"  [expiry day — BS unreliable]" if dte == 0 else ""}')

if bear_fired_final and not bull_fired_final:
    atm    = round(NIFTY_ACTUAL_OPEN / 50) * 50
    strike = atm - 50
    print(f'    Instrument : NIFTY {expiry.strftime("%d%b%Y").upper()} {strike} PE  (ATM={atm:,}, 1-OTM put)')
    print(f'    Action     : BUY up to {applicable_cap} lot(s) of 75 units each')
elif bull_fired_final and not bear_fired_final:
    atm    = round(NIFTY_ACTUAL_OPEN / 50) * 50
    strike = atm + 50
    print(f'    Instrument : NIFTY {expiry.strftime("%d%b%Y").upper()} {strike} CE  (ATM={atm:,}, 1-OTM call)')
    print(f'    Action     : BUY up to {applicable_cap} lot(s) of 75 units each')
else:
    print(f'    Action     : NO TRADE')

print(f'    Stop Loss  : Exit if premium falls {STOP_LOSS_PCT:.0%} from your entry price')
print(f'    Target     : Exit if premium rises {PROFIT_TARGET_PCT:.0%} from your entry price')
print(f'    Hard exit  : Close by 10:15 AM regardless')